# hrm-gears Twin-T4 harvest (smoke)

Kaggle **GPU T4 x2** — 2 × NVIDIA Tesla T4. Internet **On** so we can fetch the published Sudoku checkpoint from Hugging Face.

This notebook is **not** Stage 2 routing training. It inventories the cards, runs the published-weight Sudoku ACT smoke, and writes `/kaggle/working/harvest/`.

Hat tip: [Kaggle Notebooks](https://www.kaggle.com/docs/notebooks).

In [ ]:
import os, subprocess, sys
print(subprocess.check_output(["nvidia-smi"], text=True))
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), "n_gpu", torch.cuda.device_count())
assert torch.cuda.is_available(), "Accelerator is not CUDA — set Session → GPU T4 x2"
if torch.cuda.device_count() != 2:
    print("WARN: expected 2 T4s; got", torch.cuda.device_count(), "— confirm Accelerator is GPU T4 x2, not P100")

In [ ]:
from pathlib import Path
import subprocess, os

WORK = Path("/kaggle/working")
REPO = WORK / "hrm-gears"
SHA = os.environ.get("HRM_GEARS_SHA", "").strip()

if not REPO.exists():
    subprocess.check_call(["git", "clone", "https://github.com/anwhelan01/hrm-gears.git", str(REPO)])
if SHA:
    subprocess.check_call(["git", "-C", str(REPO), "checkout", SHA])
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "omegaconf", "pyyaml", "einops"])

In [ ]:
import os, subprocess, sys
from pathlib import Path
REPO = Path("/kaggle/working/hrm-gears")
sys.path.insert(0, str(REPO))
os.chdir(REPO)
rc = subprocess.call([sys.executable, "scripts/kaggle/run_session.py", "--out", "/kaggle/working"])
raise SystemExit(rc) if rc else None

In [ ]:
from pathlib import Path
h = Path("/kaggle/working/harvest")
print("harvest files:", sorted(p.name for p in h.iterdir()) if h.is_dir() else "MISSING")
print((h / "session.log").read_text() if (h / "session.log").is_file() else "no session.log")
print("Stop this session when the Output tab has harvest/. Do not leave the VM idling.")